# 02 – Baseline detector (YOLOv8n)

Trains the smallest YOLO model on the Roboflow split (kept after the leakage audit in `01_data_audit.ipynb`), then measures:

1. **Detection** on the test split: mAP and per-class precision/recall.
2. **Tray pass/fail**: false-pass and false-fail rates, with the threshold chosen on *validation* and reported on *test* (original labels and with the 3 corrected test labels).
3. **Speed**: ms per image on GPU and CPU, GFLOPs, and the trays-per-minute this allows.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Set `SMOKE_TEST = True` first (3 epochs, ~5 min) to check everything works, then `False` for the real run (~40–60 min).

In [ ]:
# Get the code and install requirements (safe to re-run)
import os, sys

REPO = "/content/MMA3001-Project-repository"
if not os.path.isdir(REPO):
    !git clone https://github.com/rhal0027/MMA3001-Project-repository.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import importlib
from google.colab import userdata
import pandas as pd
import matplotlib.pyplot as plt

from rasher_inspection import dataset as ds, evaluate as ev, benchmark as bm, train as tr, visualise as vis
for m in (ds, ev, bm, tr, vis):
    importlib.reload(m)

root = ds.download_dataset(userdata.get("ROBOFLOW_API_KEY"), "data/raw/pork-rasher", version=4)
names = ds.load_class_names(root)
data_yaml = tr.write_data_yaml(root)
print(names, data_yaml)

## 1. Train

Settings are recorded here and in `args.yaml` inside the run folder. The run is saved to Google Drive so a disconnect does not lose it; set `RETRAIN = False` to reuse a finished run.

In [ ]:
SMOKE_TEST = True          # True: 3 epochs to test the pipeline. False: the real run.
RETRAIN = True             # False: skip training and load the saved weights

CONFIG = dict(
    model="yolov8n.pt",    # smallest YOLOv8, COCO-pretrained
    epochs=3 if SMOKE_TEST else 50,
    imgsz=640,
    batch=16,
    seed=0,
    project="/content/drive/MyDrive/rasher_runs",
    name="smoke_yolov8n" if SMOKE_TEST else "baseline_yolov8n",
)
RUN_DIR = f"{CONFIG['project']}/{CONFIG['name']}"
print(CONFIG)

In [ ]:
from ultralytics import YOLO

if RETRAIN:
    tr.train_yolo(data_yaml, **CONFIG)
best = YOLO(f"{RUN_DIR}/weights/best.pt")

In [ ]:
# Training curves (loss and validation mAP per epoch)
from IPython.display import Image as Show
Show(f"{RUN_DIR}/results.png", width=900)

## 2. Detection metrics on the test split

Box-level results from Ultralytics' validator. Remember the class counts in test: unsealed 49, packaging-error 23, twisted-meat 4, loose-meat 3, wrinkle 0. Per-class numbers for the last three are not meaningful on their own.

In [ ]:
m = best.val(data=str(data_yaml), split="test", imgsz=CONFIG["imgsz"], batch=CONFIG["batch"],
             conf=0.001, project=RUN_DIR, name="test_eval", exist_ok=True, plots=True, verbose=False)

test_counts = pd.read_csv("results/class_counts.csv", index_col=0)["test"]
rows = []
for i, c in enumerate(m.box.ap_class_index):
    rows.append({"class": names[c], "test_boxes": int(test_counts.get(names[c], 0)),
                 "precision": m.box.p[i], "recall": m.box.r[i],
                 "mAP50": m.box.ap50[i], "mAP50_95": m.box.ap[i]})
rows.append({"class": "ALL", "test_boxes": int(test_counts.sum()),
             "precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50_95": m.box.map})
det = pd.DataFrame(rows).round(3)
det.to_csv(f"results/{CONFIG['name']}_test_detection.csv", index=False)
det

In [ ]:
Show(f"{RUN_DIR}/test_eval/confusion_matrix_normalized.png", width=650)

## 3. Tray pass/fail

A tray **fails** if any detection has confidence ≥ threshold. The threshold is chosen on **validation** (no false passes allowed, then fewest false fails) and applied unchanged to **test**.

Test is scored twice: with the original labels, and with the 3 missed defects found in the manual review (frames 4088, 7471, 7325) counted as defective. The 14 unlabelled *validation* images were not reviewed, so validation uses the original labels.

In [ ]:
preds = {s: tr.predict_split(best, root, s, imgsz=CONFIG["imgsz"]) for s in ("valid", "test")}
max_conf = {s: {p.image: p.max_confidence for p in preds[s]} for s in preds}

corrections = ev.load_missed_defects("results/unlabelled_review_test_seed0.csv")
truth = {
    "valid": ev.tray_truth(root, "valid"),
    "test (original labels)": ev.tray_truth(root, "test"),
    "test (corrected labels)": ev.tray_truth(root, "test", corrections),
}
for k, t in truth.items():
    print(f"{k:24s} {sum(t.values()):3d} defective, {len(t) - sum(t.values()):2d} clean")

sweep_val = ev.threshold_sweep(truth["valid"], max_conf["valid"])
chosen = ev.choose_threshold(sweep_val, max_false_pass=0.0)
if chosen is None:   # no threshold catches every validation defect: take the one with fewest false passes
    chosen = min(sweep_val, key=lambda s: (s["false_pass_rate"], s["false_fail_rate"]))
T = chosen["threshold"]
print("Chosen threshold (from validation):", T)

In [ ]:
def fmt(e):
    fp_lo, fp_hi = e["false_pass_ci"]; ff_lo, ff_hi = e["false_fail_ci"]
    return {"defective_passed": f'{e["fn"]}/{e["tp"] + e["fn"]}',
            "false_pass_rate": f'{e["false_pass_rate"]:.1%} ({fp_lo:.0%}-{fp_hi:.0%})',
            "clean_failed": f'{e["fp"]}/{e["fp"] + e["tn"]}',
            "false_fail_rate": f'{e["false_fail_rate"]:.1%} ({ff_lo:.0%}-{ff_hi:.0%})'}

tray = pd.DataFrame({k: fmt(ev.tray_errors(t, max_conf["valid" if k == "valid" else "test"], T))
                     for k, t in truth.items()}).T
tray.index.name = f"threshold {T}"
tray.to_csv(f"results/{CONFIG['name']}_tray_passfail.csv")
tray     # rates with 95% Wilson intervals

In [ ]:
# The trade-off: lowering the threshold catches more defects but rejects more good trays
fig, ax = plt.subplots(figsize=(7, 4))
for k, t in truth.items():
    sw = ev.threshold_sweep(t, max_conf["valid" if k == "valid" else "test"])
    ax.plot([s["threshold"] for s in sw], [s["false_pass_rate"] for s in sw], "-", label=f"false pass – {k}")
    ax.plot([s["threshold"] for s in sw], [s["false_fail_rate"] for s in sw], "--", label=f"false fail – {k}")
ax.axvline(T, color="grey", lw=0.8)
ax.set_xlabel("confidence threshold"); ax.set_ylabel("rate"); ax.legend(fontsize=7)
ax.set_title("Tray pass/fail trade-off")
fig.savefig(f"results/{CONFIG['name']}_threshold_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-image record, for failure analysis and reproducibility
pd.DataFrame([{"split": s, "image": p.image.name, "n_boxes": len(p.classes),
               "max_conf": round(p.max_confidence, 4),
               "defective_original": truth["valid" if s == "valid" else "test (original labels)"][p.image],
               "defective_corrected": truth["valid" if s == "valid" else "test (corrected labels)"][p.image]}
              for s in preds for p in preds[s]]).to_csv(f"results/{CONFIG['name']}_predictions.csv", index=False)

### Failure cases

Defective test trays the model passed at the chosen threshold, and clean trays it failed. Look for patterns (glare, partial trays, overlapping rashers, rare classes) and note them for the report.

In [ ]:
t_corr = truth["test (corrected labels)"]
false_pass = [img for img, d in t_corr.items() if d and max_conf["test"][img] < T]
false_fail = [img for img, d in t_corr.items() if not d and max_conf["test"][img] >= T]
print(len(false_pass), "false passes,", len(false_fail), "false fails")

def show_predictions(imgs, title):
    if not imgs:
        print("none:", title); return
    cols = min(4, len(imgs)); rows = -(-len(imgs) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, img in zip(axes.flat, imgs):
        r = best.predict(str(img), conf=T, imgsz=CONFIG["imgsz"], verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(img.name[:32], fontsize=8)
    fig.suptitle(title); fig.tight_layout(); plt.show()

show_predictions(false_pass[:12], "False passes (defective, passed) – boxes are the model's, at the threshold")
show_predictions(false_fail[:12], "False fails (clean, rejected)")

## 4. Speed

Timed per image with warm-up runs discarded, repeated 3 times. Includes Ultralytics pre- and post-processing (resize, NMS), which is what a line controller would actually wait for. Colab's GPU type varies between sessions, so the hardware is recorded with the numbers.

In [ ]:
from ultralytics.utils.torch_utils import get_flops

imgs = ds.list_images(root, "test")
speed = []
for device, n in (("cuda:0", len(imgs)), ("cpu", 20)):
    # A fresh model per device: Ultralytics keeps a model on the device of its first predict call.
    model_d = YOLO(f"{RUN_DIR}/weights/best.pt")
    s = bm.time_per_item(lambda p: model_d.predict(str(p), imgsz=CONFIG["imgsz"], device=device, verbose=False),
                         imgs[:n], warmup=5, repeats=3)
    s.update(device=str(next(model_d.model.parameters()).device),   # confirms where it actually ran
             trays_per_min=round(bm.trays_per_minute(s["median_ms"])))
    speed.append(s)

env = bm.environment()
gflops = get_flops(best.model, CONFIG["imgsz"])
assert gflops > 0, "GFLOPs is 0: the thop package is missing (pip install ultralytics-thop)"
speed_df = pd.DataFrame(speed).set_index("device").round(2)
speed_df["gpu"] = env["gpu"]
speed_df["gflops"] = round(gflops, 2)
speed_df["params_M"] = round(sum(p.numel() for p in best.model.parameters()) / 1e6, 2)
speed_df.to_csv(f"results/{CONFIG['name']}_speed.csv")
speed_df


## 5. Save results

Download the zip and attach it in the chat (or upload to `results/` on GitHub). The weights stay in Google Drive (`rasher_runs/`).

In [ ]:
!cp {RUN_DIR}/results.png results/{CONFIG['name']}_training_curves.png
!cp {RUN_DIR}/test_eval/confusion_matrix_normalized.png results/{CONFIG['name']}_confusion_matrix.png
!cd results && zip -q ../results_{CONFIG['name']}.zip {CONFIG['name']}_*
from google.colab import files
files.download(f"results_{CONFIG['name']}.zip")
